# Multilingual Language Identification with Recurrent Neural Networks

**Alex Eagles Tech — NLP & Intelligent Systems Unit**

This notebook builds and evaluates a recurrent neural architecture (a
bidirectional LSTM) that identifies the natural language of a short, raw
text string. The pipeline covers:

1. Data loading and inspection (`papluca/language-identification`)
2. A custom whitespace/punctuation-aware tokenizer and vocabulary
3. PyTorch `Dataset` / `DataLoader` construction for train/validation/test
4. A BiLSTM classification model with an embedding layer, dropout
   regularization, and a linear classification head
5. Training with per-epoch loss/accuracy tracking
6. Final evaluation on the held-out 10,000-sample test split
7. A standalone inference function for live, single-message predictions

> **Note on environment:** this notebook expects internet access to the
> Hugging Face Hub (to download the dataset) and a working PyTorch
> installation. Run it in an environment such as Google Colab, or locally
> after `pip install torch datasets scikit-learn matplotlib pandas`.


## 1. Setup

In [ ]:
# If running in a fresh environment (e.g. Colab), uncomment the line below:
# !pip install -q datasets torch scikit-learn matplotlib pandas

import re
import math
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## 2. Data Pipeline

### 2.1 Load the dataset

`papluca/language-identification` ships with `train` (70k rows),
`validation` (10k rows) and `test` (10k rows) splits, each containing a
`text` field and a `labels` field (an ISO 639-1 language code across 20
languages).


In [ ]:
dataset = load_dataset("papluca/language-identification")
print(dataset)


In [ ]:
train_df = dataset["train"].to_pandas()
val_df = dataset["validation"].to_pandas()
test_df = dataset["test"].to_pandas()

print(f"Train: {len(train_df):,} rows | Validation: {len(val_df):,} rows | Test: {len(test_df):,} rows")
train_df.head()


### 2.2 Inspect class balance across the 20 target languages

In [ ]:
label_counts = train_df["labels"].value_counts().sort_index()
print(label_counts)

fig, ax = plt.subplots(figsize=(10, 5))
label_counts.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Training set: samples per language")
ax.set_xlabel("Language code")
ax.set_ylabel("Number of samples")
plt.tight_layout()
plt.show()

print(f"\nNumber of distinct languages: {train_df['labels'].nunique()}")
print(f"Min class count: {label_counts.min()} | Max class count: {label_counts.max()}")


The dataset is (by construction) close to perfectly balanced across all
20 languages, which means plain accuracy is a reliable evaluation metric
here (no class-imbalance correction is required).

### 2.3 Tokenization pipeline

We build a simple, dependency-free whitespace-and-punctuation tokenizer,
then cap the vocabulary at the **top 40,000 tokens** by frequency in the
training split. Two indices are reserved for special tokens:

- `0` → `<PAD>` (padding token)
- `1` → `<UNK>` (out-of-vocabulary / unknown token)


In [ ]:
PAD_IDX = 0
UNK_IDX = 1
MAX_VOCAB_SIZE = 40_000
MAX_LEN = 64

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tokenize(text: str):
    """Lowercase + split into word / punctuation tokens (language-agnostic, \w
    matches Unicode letters so this works across scripts, e.g. Cyrillic, Arabic,
    CJK characters are each captured as individual tokens by \w)."""
    return TOKEN_RE.findall(text.lower())


def build_vocab(texts, max_vocab_size=MAX_VOCAB_SIZE):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))

    most_common = counter.most_common(max_vocab_size)
    vocab = {"<PAD>": PAD_IDX, "<UNK>": UNK_IDX}
    for tok, _ in most_common:
        if tok not in vocab:
            vocab[tok] = len(vocab)
    return vocab


vocab = build_vocab(train_df["text"].tolist())
print(f"Vocabulary size (incl. special tokens): {len(vocab):,}")


In [ ]:
def encode(text: str, vocab: dict, max_len: int = MAX_LEN):
    tokens = tokenize(text)
    ids = [vocab.get(tok, UNK_IDX) for tok in tokens][:max_len]
    length = len(ids)
    if length < max_len:
        ids = ids + [PAD_IDX] * (max_len - length)
    return ids, length


# Sanity check
sample_text = train_df.iloc[0]["text"]
sample_ids, sample_len = encode(sample_text, vocab)
print("Sample text:", sample_text[:80])
print("Encoded (first 20 ids):", sample_ids[:20])
print("True token length (pre-padding):", sample_len)


### 2.4 Label encoding

In [ ]:
label_encoder = LabelEncoder()
label_encoder.fit(train_df["labels"])
num_classes = len(label_encoder.classes_)
print(f"Number of classes: {num_classes}")
print("Classes:", list(label_encoder.classes_))


### 2.5 PyTorch `Dataset` and `DataLoader` construction

In [ ]:
class LanguageIDDataset(Dataset):
    def __init__(self, texts, labels, vocab, label_encoder, max_len=MAX_LEN):
        self.texts = texts
        self.labels = label_encoder.transform(labels)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids, length = encode(self.texts[idx], self.vocab, self.max_len)
        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(length, dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long),
        )


train_ds = LanguageIDDataset(train_df["text"].tolist(), train_df["labels"].tolist(), vocab, label_encoder)
val_ds = LanguageIDDataset(val_df["text"].tolist(), val_df["labels"].tolist(), vocab, label_encoder)
test_ds = LanguageIDDataset(test_df["text"].tolist(), test_df["labels"].tolist(), vocab, label_encoder)

BATCH_SIZE = 128

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


## 3. Architecture Design

We use a **bidirectional LSTM** classifier:

- **Embedding layer** (`vocab_size x embed_dim`, `padding_idx=0`) projects
  token ids into a dense continuous space and ignores the padding token's
  gradient contribution.
- **Packed bidirectional LSTM** models temporal order in both directions,
  which helps disambiguate short, ambiguous phrases where language cues
  can appear anywhere in the sequence.
- The final forward/backward hidden states are concatenated and passed
  through a **dropout-regularized MLP head** with a **ReLU** non-linearity
  before the final linear classification layer.
- Trained with **Cross-Entropy Loss**.


In [ ]:
class BiLSTMLanguageClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_classes,
        embed_dim=128,
        hidden_dim=128,
        num_layers=1,
        dropout=0.3,
        pad_idx=PAD_IDX,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embed_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, input_ids, lengths):
        embedded = self.embedding(input_ids)          # (B, L, E)
        embedded = self.embed_dropout(embedded)

        lengths_clamped = lengths.clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths_clamped, batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)

        # h_n: (num_layers * 2, B, hidden_dim) -> take the last layer's
        # forward and backward hidden states and concatenate them.
        h_forward = h_n[-2]
        h_backward = h_n[-1]
        final_hidden = torch.cat([h_forward, h_backward], dim=1)  # (B, hidden_dim*2)

        logits = self.classifier(final_hidden)
        return logits


model = BiLSTMLanguageClassifier(
    vocab_size=len(vocab),
    num_classes=num_classes,
    embed_dim=128,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3,
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")


## 4. Model Training & Evaluation

### 4.1 Training / evaluation loop helpers


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


def run_epoch(loader, model, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_correct, total_samples = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for input_ids, lengths, labels in loader:
            input_ids, lengths, labels = input_ids.to(device), lengths.to(device), labels.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(input_ids, lengths)
            loss = criterion(logits, labels)

            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            batch_size = labels.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == labels).sum().item()
            total_samples += batch_size

    return total_loss / total_samples, total_correct / total_samples


### 4.2 Train for multiple epochs, tracking loss and accuracy

In [ ]:
NUM_EPOCHS = 8

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()

    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer)
    val_loss, val_acc = run_epoch(val_loader, model, criterion, optimizer=None)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    elapsed = time.time() - start
    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"{elapsed:.1f}s"
    )


### 4.3 Plot training curves

In [ ]:
epochs_range = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(epochs_range, history["train_loss"], label="Train loss", marker="o")
axes[0].plot(epochs_range, history["val_loss"], label="Validation loss", marker="o")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Cross-entropy loss")
axes[0].set_title("Loss per epoch")
axes[0].legend()

axes[1].plot(epochs_range, history["train_acc"], label="Train accuracy", marker="o")
axes[1].plot(epochs_range, history["val_acc"], label="Validation accuracy", marker="o")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Accuracy per epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


### 4.4 Final evaluation on the held-out 10,000-sample test set

In [ ]:
test_loss, test_acc = run_epoch(test_loader, model, criterion, optimizer=None)
print(f"Final test loss: {test_loss:.4f}")
print(f"Final test accuracy: {test_acc:.4f}")


In [ ]:
# Detailed per-class report + confusion matrix
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for input_ids, lengths, labels in test_loader:
        input_ids, lengths = input_ids.to(device), lengths.to(device)
        logits = model(input_ids, lengths)
        preds = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

target_names = label_encoder.classes_
print(classification_report(all_labels, all_preds, target_names=target_names, digits=3))


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(num_classes))
ax.set_yticks(range(num_classes))
ax.set_xticklabels(target_names, rotation=90)
ax.set_yticklabels(target_names)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix — test set")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 5. Custom Inference Pipeline

A standalone function that takes an arbitrary raw text string (e.g. a
live customer message), tokenizes and pads it exactly as during training,
runs a forward pass, and returns:

- the predicted language code, and
- the full class-probability distribution (confidence scores).


In [ ]:
def predict_language(text: str, model, vocab, label_encoder, max_len=MAX_LEN, top_k=5):
    model.eval()
    ids, length = encode(text, vocab, max_len)

    input_ids = torch.tensor([ids], dtype=torch.long).to(device)
    lengths = torch.tensor([length], dtype=torch.long).to(device)

    with torch.no_grad():
        logits = model(input_ids, lengths)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_idx = int(np.argmax(probs))
    pred_label = label_encoder.inverse_transform([pred_idx])[0]

    ranked = sorted(
        zip(label_encoder.classes_, probs), key=lambda x: x[1], reverse=True
    )[:top_k]
    confidence_scores = {lang: float(p) for lang, p in ranked}

    return {
        "text": text,
        "predicted_language": pred_label,
        "confidence": float(probs[pred_idx]),
        "top_k_scores": confidence_scores,
    }


In [ ]:
# Demo: a few short / ambiguous / multilingual example messages
demo_messages = [
    "Bonjour, pouvez-vous m'aider avec ma commande ?",
    "Hola, necesito ayuda con mi pedido por favor.",
    "Can you confirm the transaction reference number for my account?",
    "Guten Tag, ich habe eine Frage zu meiner Rechnung.",
    "Ok",
    "merci",
]

for msg in demo_messages:
    result = predict_language(msg, model, vocab, label_encoder)
    print(f"Text: {result['text']!r}")
    print(f"  -> Predicted: {result['predicted_language']} (confidence: {result['confidence']:.3f})")
    print(f"  -> Top-{len(result['top_k_scores'])} scores: {result['top_k_scores']}")
    print()


## 6. Summary

- Built a whitespace/punctuation tokenizer and a 40k-token capped
  vocabulary directly from the training split, with reserved indices for
  `<PAD>` (0) and `<UNK>` (1).
- Standardized all sequences to `max_len = 64` via truncation and padding.
- Trained a bidirectional LSTM classifier (embedding → BiLSTM → dropout-
  regularized MLP head → softmax over 20 languages) with Cross-Entropy
  loss and the Adam optimizer.
- Tracked per-epoch training/validation loss and accuracy, then evaluated
  the final model on the untouched 10,000-sample test split, reporting
  test loss, test accuracy, a full classification report, and a confusion
  matrix.
- Implemented `predict_language(...)`, a standalone inference function
  for arbitrary live text, returning the predicted language and
  confidence scores across the top candidate languages — directly usable
  as the core of an automated message-routing service.

### Next steps for production
- Swap the from-scratch tokenizer for a subword tokenizer (e.g. byte-pair
  encoding) to reduce `<UNK>` collisions on rare tokens and unseen
  scripts.
- Add mixed-precision training and ONNX/TorchScript export to meet the
  low-latency requirement at inference time.
- Track per-language precision/recall over time in production to catch
  drift on the languages your customer base uses most.
